# Analysis outline

See [README.md](README.md).

# Set up

Run the following cell.  It loads some required functions.

In [18]:
run kondrashov

# Search for human sequences

Below we try to retrieve human sequences for each gene programatically.  We
choose the [RefSeq Select](https://www.ncbi.nlm.nih.gov/refseq/refseq_select/) database.

In [2]:
# genes in Kondrashov et al. (2002) are already listed in kondrashov.py
print(loci)

['ABCD1', 'ALPL', 'AR', 'ATP7B', 'BTK', 'CASR', 'CBS', 'CFTR', 'CYBB', 'F7', 'F8', 'F9', 'G6PD', 'GALT', 'GBA1', 'GJB1', 'HBB', 'HPRT1', 'IL2RG', 'KCNH2', 'KCNQ1', 'L1CAM', 'LDLR', 'MPZ', 'MYH7', 'TYR', 'PAH', 'PMM2', 'RHO', 'TP53', 'TTR', 'VWF']


In [3]:
genes = {}
for locus in loci:
    query = f"Homo sapiens[Organism] AND {locus}[Gene Name] AND refseq_select[Filter]"
    handle = Entrez.esearch(db="nucleotide", term=query, retmax=100)
    record = Entrez.read(handle)
    handle.close()
    print(f"{locus}: {record['Count']} ->", record['IdList'][:4])
    genes.update({locus: record['IdList']})

ABCD1: 1 -> ['1519313321']
ALPL: 1 -> ['1519315936']
AR: 3 -> ['1654124212', '1519245710', '1519243240']
ATP7B: 1 -> ['1677498587']
BTK: 1 -> ['1780222528']
CASR: 1 -> ['1677537479']
CBS: 1 -> ['1780222567']
CFTR: 1 -> ['1732746288']
CYBB: 1 -> ['1732746192']
F7: 1 -> ['1779521762']
F8: 1 -> ['1812229661']
F9: 1 -> ['1732746198']
G6PD: 1 -> ['1497290737']
GALT: 1 -> ['1677502190']
GBA1: 1 -> ['1519244100']
GJB1: 1 -> ['1606717073']
HBB: 1 -> ['1401724401']
HPRT1: 1 -> ['1519243265']
IL2RG: 1 -> ['1780222514']
KCNH2: 1 -> ['1732746325']
KCNQ1: 1 -> ['1732746161']
L1CAM: 1 -> ['1653961419']
LDLR: 1 -> ['1732746181']
MPZ: 1 -> ['1519245315']
MYH7: 1 -> ['1519242569']
TYR: 1 -> ['1519243869']
PAH: 1 -> ['1519244862']
PMM2: 1 -> ['1519243247']
RHO: 2 -> ['169808383', '1519316345']
TP53: 1 -> ['1808862652']
TTR: 1 -> ['1780222569']
VWF: 1 -> ['1813372008']


Two of the genes, AR and RHO, have more than one RefSeq Select transcripts.
Let's look at them more closely.

* AR: only the first sequence is valid.

* RHO: only the second sequence is valid.

In [155]:
# AR
for i in genes['AR']:
    stream = Entrez.efetch(db="nucleotide", id=i, rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    print(f"ID: {i}")
    print(f"{record2.id}: {record2.description}\nLength: {len(record2.seq)} nt\n")

ID: 1654124212
NM_000044.6: Homo sapiens androgen receptor (AR), transcript variant 1, mRNA
Length: 10667 nt

ID: 1519245710
NM_001657.4: Homo sapiens amphiregulin (AREG), mRNA
Length: 1234 nt

ID: 1519243240
NM_001628.4: Homo sapiens aldo-keto reductase family 1 member B (AKR1B1), transcript variant 1, mRNA
Length: 1361 nt



In [156]:
# remove wrong sequences
del genes['AR'][2]
del genes['AR'][1]
genes['AR']

['1654124212']

In [157]:
# RHO
for i in genes['RHO']:
    stream = Entrez.efetch(db="nucleotide", id=i, rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    print(f"ID: {i}")
    print(f"{record2.id}: {record2.description}\nLength: {len(record2.seq)} nt\n")

ID: 1519316345
NM_014578.4: Homo sapiens ras homolog family member D (RHOD), transcript variant 1, mRNA
Length: 1104 nt

ID: 169808383
NM_000539.3: Homo sapiens rhodopsin (RHO), mRNA
Length: 2768 nt



In [101]:
# remove wrong sequence
del genes['RHO'][0]
genes['RHO']

['169808383']

In [4]:
for gene in genes:
    stream = Entrez.efetch(db="nucleotide", id=genes[gene][0], rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    genes[gene].append(record2.id)
    print(gene, genes[gene])

ABCD1 ['1519313321', 'NM_000033.4']
ALPL ['1519315936', 'NM_000478.6']
AR ['1654124212', '1519245710', '1519243240', 'NM_000044.6']
ATP7B ['1677498587', 'NM_000053.4']
BTK ['1780222528', 'NM_000061.3']
CASR ['1677537479', 'NM_000388.4']
CBS ['1780222567', 'NM_000071.3']
CFTR ['1732746288', 'NM_000492.4']
CYBB ['1732746192', 'NM_000397.4']
F7 ['1779521762', 'NM_019616.4']
F8 ['1812229661', 'NM_000132.4']
F9 ['1732746198', 'NM_000133.4']
G6PD ['1497290737', 'NM_001360016.2']
GALT ['1677502190', 'NM_000155.4']
GBA1 ['1519244100', 'NM_000157.4']
GJB1 ['1606717073', 'NM_000166.6']
HBB ['1401724401', 'NM_000518.5']
HPRT1 ['1519243265', 'NM_000194.3']
IL2RG ['1780222514', 'NM_000206.3']
KCNH2 ['1732746325', 'NM_000238.4']
KCNQ1 ['1732746161', 'NM_000218.3']
L1CAM ['1653961419', 'NM_001278116.2']
LDLR ['1732746181', 'NM_000527.5']
MPZ ['1519245315', 'NM_000530.8']
MYH7 ['1519242569', 'NM_000257.4']
TYR ['1519243869', 'NM_000372.5']
PAH ['1519244862', 'NM_000277.3']
PMM2 ['1519243247', 'NM_0003

In [141]:
# pull protein ID
handle_link = Entrez.elink(dbfrom="nuccore", db="protein", id="NM_000552.5")
link_record = Entrez.read(handle_link)
protein_id = link_record[0]["LinkSetDb"][0]["Link"][0]["Id"]
protein_id

'1813372009'

In [ ]:
# pull protein sequence ID
stream = Entrez.efetch(db="protein", id='1813372009', rettype="gb", retmode="text")
record2 = SeqIO.read(stream, "genbank")
stream.close()
record2.id

'NP_000543.3'

# Search for orthologs

Primates are represented by [Registry Number
txid9443](https://www.ncbi.nlm.nih.gov/mesh/68011323).

Searching in the `Gene` database appears to retrieve the sequences that can be
obtained by doing the following:

* In the [NCBI query page](https://www.ncbi.nlm.nih.gov/labs/gquery/) search for
  `Homo sapiens {Gene Name}` (remove braces).  

* When the search results come up click on `Orthologs`.  

* Under `Selected taxa` write **Primates**. 

This was the procedure used in Hannah Esopenko's analysis.

In [5]:
orth = {}
for locus in loci:
    query = f"{locus}[Gene Name] AND txid9443[Organism]"
    handle = Entrez.esearch(db="gene", term=query, retmax=100)
    record = Entrez.read(handle)
    handle.close()
    print(f"{locus}: {record['Count']} ->", record['IdList'][:4])
    orth.update({locus: record['IdList']})

ABCD1: 35 -> ['215', '6367', '696794', '465923']
ALPL: 35 -> ['249', '717809', '456600', '100401643']
AR: 40 -> ['367', '374', '231', '574293']
ATP7B: 34 -> ['540', '713781', '452734', '101866907']
BTK: 34 -> ['695', '703000', '465759', '102123534']
CASR: 35 -> ['846', '714441', '460628', '101867346']
CBS: 32 -> ['875', '736626', '100423684', '100414619']
CFTR: 37 -> ['1080', '574346', '463674', '100137035']
CYBB: 34 -> ['1536', '696542', '465566', '102122418']
F7: 32 -> ['2155', '721933', '744785', '100410381']
F8: 33 -> ['2157', '100424151', '473838', '100393161']
F9: 34 -> ['2158', '695578', '465887', '102146175']
G6PD: 33 -> ['2539', '701137', '743041', '100388810']
GALT: 38 -> ['2592', '145173', '473219', '106993709']
GBA1: 29 -> ['2629', '449571', '719103', '138390798']
GJB1: 35 -> ['2705', '706327', '465696', '102128042']
HBB: 25 -> ['3043', '450978', '101926697', '715559']
HPRT1: 34 -> ['3251', '735894', '101867079', '709186']
IL2RG: 36 -> ['3561', '641338', '101059843', '10214

In [13]:
print(record)
#dir(record)
print(record.keys())
#print(f"{locus}: {record['Count']} ->", record['IdList'][:4])
#orth.update({locus: record['IdList']})

{'Count': '35', 'RetMax': '35', 'RetStart': '0', 'IdList': ['7450', '722019', '451773', '100434817', '100396684', '138397437', '129483487', '129009569', '128561143', '126930424', '123640137', '117090655', '116534262', '116472569', '112634393', '111555950', '108536508', '108316350', '105869041', '105805087', '105719737', '105583007', '105541774', '105521211', '104680130', '103271056', '103218442', '102132631', '101142333', '101044155', '101011576', '100985238', '100958412', '100580759', '105484247'], 'TranslationSet': [], 'TranslationStack': [{'Term': 'VWF[Gene Name]', 'Field': 'Gene Name', 'Count': '821', 'Explode': 'N'}, {'Term': 'txid9443[Organism]', 'Field': 'Organism', 'Count': '1903822', 'Explode': 'Y'}, 'AND'], 'QueryTranslation': 'VWF[Gene Name] AND txid9443[Organism]'}
dict_keys(['Count', 'RetMax', 'RetStart', 'IdList', 'TranslationSet', 'TranslationStack', 'QueryTranslation'])


In [ ]:
for locus in loci:
    query = f"{locus}[Gene Name] AND txid9443[Organism]"
    
    # Step 1: Search for gene IDs
    handle = Entrez.esearch(db="gene", term=query, retmax=100)
    record = Entrez.read(handle)
    handle.close()
    
    gene_ids = record["IdList"]
    if not gene_ids:
        print(f"No results for {locus}")
    print(f"{locus}: {record['Count']}", gene_ids) 
#        continue
'''
    # Step 2: Fetch gene details to get linked protein accessions
    for gene_id in gene_ids:
        # Fetch gene record in XML format
        handle = Entrez.efetch(db="gene", id=gene_id, rettype="gene_table", retmode="xml")
        gene_record = Entrez.read(handle)
        handle.close()
    

        # Step 3: Use elink to find linked protein records
        link_handle = Entrez.elink(dbfrom="gene", db="protein", id=gene_id)
        link_record = Entrez.read(link_handle)
        link_handle.close()

        protein_ids = []
        if link_record[0]["LinkSetDb"]:
            for link in link_record[0]["LinkSetDb"][0]["Link"]:
                protein_ids.append(link["Id"])

        # Step 4: Fetch protein accession numbers
        protein_accessions = []
        for protein_id in protein_ids:
            prot_handle = Entrez.efetch(db="protein", id=protein_id, rettype="acc", retmode="text")
            accession = prot_handle.read().strip()
            prot_handle.close()
            protein_accessions.append(accession)

        print(f"Locus: {locus} | Gene ID: {gene_id} | Protein Accessions: {protein_accessions}")
        '''

ABCD1: 35 ['215', '6367', '696794', '465923', '102120996', '100409370', '138379168', '129475142', '129023948', '128576678', '126945380', '123628458', '117078183', '116812184', '116541212', '112615076', '108535081', '108294745', '105867309', '105824530', '105707094', '105598745', '105555352', '105521622', '104661178', '103251037', '103232990', '101125363', '101029873', '101015478', '100974677', '100947429', '100586126', '100441172', '105467327']
ALPL: 35 ['249', '717809', '456600', '100401643', '101926551', '138390496', '129472619', '129056640', '129011926', '128575315', '126955155', '123635879', '117093169', '116813037', '116566413', '112604796', '111547722', '108538443', '108288034', '105873619', '105815701', '105708598', '105595032', '105546845', '105505986', '104663214', '103265021', '103225436', '101134160', '101052916', '101001000', '100987532', '100952679', '100602436', '105483075']
AR: 40 ['367', '374', '231', '574293', '102117720', '747460', '737737', '100401612', '138378525', 

'\n    # Step 2: Fetch gene details to get linked protein accessions\n    for gene_id in gene_ids:\n        # Fetch gene record in XML format\n        handle = Entrez.efetch(db="gene", id=gene_id, rettype="gene_table", retmode="xml")\n        gene_record = Entrez.read(handle)\n        handle.close()\n    print(gene_record)\n\n\n        # Step 3: Use elink to find linked protein records\n        link_handle = Entrez.elink(dbfrom="gene", db="protein", id=gene_id)\n        link_record = Entrez.read(link_handle)\n        link_handle.close()\n\n        protein_ids = []\n        if link_record[0]["LinkSetDb"]:\n            for link in link_record[0]["LinkSetDb"][0]["Link"]:\n                protein_ids.append(link["Id"])\n\n        # Step 4: Fetch protein accession numbers\n        protein_accessions = []\n        for protein_id in protein_ids:\n            prot_handle = Entrez.efetch(db="protein", id=protein_id, rettype="acc", retmode="text")\n            accession = prot_handle.read().stri

In [65]:
print(gene_ids)

['7450', '722019', '451773', '100434817', '100396684', '138397437', '129483487', '129009569', '128561143', '126930424', '123640137', '117090655', '116534262', '116472569', '112634393', '111555950', '108536508', '108316350', '105869041', '105805087', '105719737', '105583007', '105541774', '105521211', '104680130', '103271056', '103218442', '102132631', '101142333', '101044155', '101011576', '100985238', '100958412', '100580759', '105484247']


In [27]:
# Step 2: Fetch gene details to get linked protein accessions
for gene_id in gene_ids:
    # Fetch gene record in XML format
    handle = Entrez.efetch(db="gene", id=gene_id, rettype="gene_table", retmode="xml")
    gene_record = Entrez.read(handle)
    handle.close()
print(gene_record)

[{'Entrezgene_track-info': {'Gene-track': {'Gene-track_geneid': '105484247', 'Gene-track_status': StringElement('0', attributes={'value': 'live'}), 'Gene-track_create-date': {'Date': {'Date_std': {'Date-std': {'Date-std_year': '2015', 'Date-std_month': '4', 'Date-std_day': '1'}}}}, 'Gene-track_update-date': {'Date': {'Date_std': {'Date-std': {'Date-std_year': '2026', 'Date-std_month': '4', 'Date-std_day': '8', 'Date-std_hour': '16', 'Date-std_minute': '2', 'Date-std_second': '36'}}}}}}, 'Entrezgene_type': StringElement('6', attributes={'value': 'protein-coding'}), 'Entrezgene_source': {'BioSource': {'BioSource_genome': StringElement('1', attributes={'value': 'genomic'}), 'BioSource_origin': StringElement('1', attributes={'value': 'natural'}), 'BioSource_org': {'Org-ref': {'Org-ref_taxname': 'Macaca nemestrina', 'Org-ref_common': 'pig-tailed macaque', 'Org-ref_db': [{'Dbtag_db': 'taxon', 'Dbtag_tag': {'Object-id': {'Object-id_id': '9545'}}}], 'Org-ref_orgname': {'OrgName': {'OrgName_nam

In [43]:
print(gene_record[0].keys())
print(gene_record[0]['Entrezgene_gene']['Gene-ref']['Gene-ref_locus'])
gene_record
print(gene_record[0]['Entrezgene_gene']['Gene-ref']['Gene-ref_locus'])
print(gene_record[0]['Entrezgene_prot'])


dict_keys(['Entrezgene_track-info', 'Entrezgene_type', 'Entrezgene_source', 'Entrezgene_gene', 'Entrezgene_prot', 'Entrezgene_gene-source', 'Entrezgene_locus', 'Entrezgene_properties', 'Entrezgene_comments', 'Entrezgene_unique-keys', 'Entrezgene_xtra-index-terms'])
LOC105484247
LOC105484247
{'Prot-ref': {'Prot-ref_desc': 'von Willebrand factor'}}


In [45]:
# Step 3: Use elink to find linked protein records
link_handle = Entrez.elink(dbfrom="gene", db="protein", id=gene_id)
link_record = Entrez.read(link_handle)
link_handle.close()

In [57]:
print(link_record)
print(link_record[0].keys())
print(link_record[0]['LinkSetDb'][0]['Link'])
print(link_record[0]['LinkSetDb'][0])

[{'LinkSetDb': [{'Link': [{'Id': '2898514581'}, {'Id': '2898514577'}], 'DbTo': 'protein', 'LinkName': 'gene_protein'}, {'Link': [{'Id': '2898514581'}, {'Id': '2898514577'}], 'DbTo': 'protein', 'LinkName': 'gene_protein_refseq'}], 'ERROR': [], 'LinkSetDbHistory': [], 'DbFrom': 'gene', 'IdList': ['105484247']}]
dict_keys(['LinkSetDb', 'ERROR', 'LinkSetDbHistory', 'DbFrom', 'IdList'])
[{'Id': '2898514581'}, {'Id': '2898514577'}]
{'Link': [{'Id': '2898514581'}, {'Id': '2898514577'}], 'DbTo': 'protein', 'LinkName': 'gene_protein'}


In [63]:
protein_ids = []
if link_record[0]["LinkSetDb"]:
    for link in link_record[0]["LinkSetDb"][0]["Link"]:
        protein_ids.append(link["Id"])
print(len(protein_ids))
print(protein_ids)

2
['2898514581', '2898514577']


In [64]:
# Step 4: Fetch protein accession numbers
protein_accessions = []
for protein_id in protein_ids:
    prot_handle = Entrez.efetch(db="protein", id=protein_id, rettype="acc", retmode="text")
    accession = prot_handle.read().strip()
    prot_handle.close()
    protein_accessions.append(accession)

print(f"Locus: {locus} | Gene ID: {gene_id} | Protein Accessions: {protein_accessions}")

Locus: VWF | Gene ID: 105484247 | Protein Accessions: ['XP_070928431.1', 'XP_070928430.1']


In [ ]:
# GBA1: 29 -> ['2629', '449571', '719103', '138390798']
# the following pulls the 4th record
handle = Entrez.efetch(db="gene", id='138390798', rettype="xml", retmode="text")
record = x2d.parse(handle.read().decode('utf-8'))
handle.close()
record

The code below shows where the species, and transcript and protein IDs are located.

In [190]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_source']['BioSource']['BioSource_org']['Org-ref']['Org-ref_taxname']

'Eulemur rufifrons'

In [167]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary']['Gene-commentary_type']['@value']

'mRNA'

In [168]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary']['Gene-commentary_accession']

'XM_069480485'

In [169]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary']['Gene-commentary_version']

'1'

In [179]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary']['Gene-commentary_products']['Gene-commentary']['Gene-commentary_type']['@value']

'peptide'

In [ ]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary']['Gene-commentary_products']['Gene-commentary']['Gene-commentary_accession']

'XP_069336586'

In [178]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary']['Gene-commentary_products']['Gene-commentary']['Gene-commentary_version']

'1'

In [191]:
# GBA1: 29 -> ['2629', '449571', '719103', '138390798']
# the following pulls the third record
handle = Entrez.efetch(db="gene", id='719103', rettype="xml", retmode="text")
record = x2d.parse(handle.read().decode('utf-8'))
handle.close()
record

In [192]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_source']['BioSource']['BioSource_org']['Org-ref']['Org-ref_taxname']

'Macaca mulatta'

In [202]:
len(record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary'])

8

In [210]:
print("mRNA accessions")
for i in range(8):
    acc = record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary'][i]['Gene-commentary_accession']
    ver = record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary'][i]['Gene-commentary_version']
    print(f"{acc}.{ver}")

mRNA accessions
XM_077943636.1
XM_077943637.1
XM_077943641.1
XM_077943643.1
XM_015111268.3
XM_015111283.3
XM_028828334.2
XM_028828341.2


In [211]:
print("protein accessions")
for i in range(8):
    acc = record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary'][i]['Gene-commentary_products']['Gene-commentary']['Gene-commentary_accession']
    ver = record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary']['Gene-commentary_products']['Gene-commentary'][i]['Gene-commentary_products']['Gene-commentary']['Gene-commentary_version']
    print(f"{acc}.{ver}")

protein accessions
XP_077799762.1
XP_077799763.1
XP_077799767.1
XP_077799769.1
XP_014966754.3
XP_014966769.3
XP_028684167.2
XP_028684174.2


# Reading from ClinVar database

In [145]:
handle = Entrez.efetch(db="clinvar", id='VCV000851448', rettype='vcv', is_varationid=True, from_esearch=True)
tmp = x2d.parse(handle.read().decode('utf-8'))
handle.close()
tmp

{'ClinVarResult-Set': {'set': None}}

In [14]:
run Kondrashov

In [11]:
get_variant_ids('ADAR')

(1756,
 ['4852057', '4821179', '4819135', '4813819', '4794958', '4794911', '4794888', '4794849', '4794838', '4794763', '4794757', '4794722', '4794499', '4794227', '4794094', '4794020', '4793992', '4793567', '4793423', '4793409', '4793170', '4793047', '4793010', '4792931', '4792911', '4792863', '4792861', '4792828', '4792733', '4792717', '4792607', '4792601', '4792554', '4792509', '4792423', '4792334', '4792324', '4792095', '4791928', '4791864', '4791846', '4791722', '4791658', '4791505', '4791465', '4791104', '4791068', '4790914', '4790829', '4790587', '4790537', '4790488', '4790237', '4790208', '4790104', '4790055', '4790025', '4789843', '4789808', '4789141', '4788867', '4788766', '4788714', '4788504', '4788498', '4788367', '4787947', '4787692', '4787523', '4787393', '4787331', '4787252', '4787176', '4787086', '4787019', '4786969', '4786905', '4786887', '4786586', '4786540', '4786448', '4786358', '4786348', '4786287', '4786268', '4786131', '4786108', '4786056', '4786005', '4785998', '

In [ ]:
rec = get_variant('4849478')
rec

In [16]:
for locus in loci:
    variants = get_variant_ids(locus)
    print(locus, variants, len(variants))

ABCD1 (2061, ['4852056', '4851495', '4851269', '4851184', '4851137', '4849478', '4840115', '4822625', '4819835', '4819267', '4819121', '4818089', '4818088', '4809010', '4807940', '4807524', '4806445', '4802208', '4802207', '4802205', '4802204', '4802203', '4802202', '4802201', '4802200', '4802199', '4802198', '4796070', '4795941', '4778430', '4776436', '4774687', '4773396', '4764160', '4756066', '4755890', '4755481', '4754592', '4752726', '4749162', '4744815', '4744352', '4737692', '4736820', '4736681', '4736175', '4735714', '4735020', '4734900', '4732327', '4732184', '4731701', '4731286', '4730515', '4729930', '4723074', '4721692', '4719857', '4719764', '4717975', '4715988', '4715948', '4714903', '4714702', '4713904', '4713550', '4711999', '4710909', '4710908', '4710906', '4710905', '4710903', '4710901', '4710858', '4707858', '4705633', '4705094', '4696104', '4692962', '4689902', '4687895', '4687823', '4686130', '4683026', '4564388', '4564361', '4539014', '4530964', '4528200', '452690